# Music Analysis Pipeline

곡 하나를 넣으면 다음 세 가지를 자동으로 수행합니다:

1. **음고(피치) 변화 추출** — 시간별 f0 궤적
2. **섹션 분할** — 벌스/코러스 등 구조적 경계 감지
3. **분위기 전환부 감지 + 섹션별 분위기 분석** — valence/energy/brightness/tension

지원 포맷: `.wav`, `.flac`, `.mp3` (librosa + audioread/soundfile 경유).

## 사용법

```python
result = analyze_track("/path/to/song.mp3")
```

`result`는 세 분석 결과를 모두 담은 dict입니다. 아래 셀들을 순서대로 실행한 뒤 마지막 셀에서 파일 경로만 바꿔 돌리면 됩니다.

## 0. 의존성

처음 한 번만 실행하세요. mp3 디코딩을 위해 ffmpeg가 시스템에 설치되어 있어야 합니다 (`brew install ffmpeg`).

In [ ]:
# %pip install -q librosa soundfile numpy scipy matplotlib

In [ ]:
import os
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple

import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter

SUPPORTED_EXT = {".wav", ".flac", ".mp3"}
SR = 22050  # 분석 샘플레이트 (librosa 기본). 정확도보다 일관성 우선.

## 1. 로딩

확장자 검사 후 모노로 리샘플링해 로드합니다. 스테레오는 분석에 불필요하므로 평균으로 다운믹스.

In [ ]:
def load_audio(path: str, sr: int = SR) -> Dict:
    ext = os.path.splitext(path)[1].lower()
    if ext not in SUPPORTED_EXT:
        raise ValueError(f"Unsupported format: {ext}. Use one of {SUPPORTED_EXT}")
    y_stereo, sr_out = librosa.load(path, sr=sr, mono=False)
    if y_stereo.size == 0:
        raise ValueError(f"Empty audio: {path}")
    # 모노 파일이면 동일 신호를 양 채널에 복제
    if y_stereo.ndim == 1:
        y_stereo = np.stack([y_stereo, y_stereo])
    return {
        "left": y_stereo[0],
        "right": y_stereo[1],
        "mono": (y_stereo[0] + y_stereo[1]) / 2.0,
        "sr": sr_out,
    }

## 2. 음고(피치) 변화 추출

`librosa.pyin`으로 단성 f0를 추정합니다. 팝/가창 트랙에서 보컬 라인을 근사하기 좋은 방법. 다성악 믹스에서는 지배적인 음고 정도로 해석하세요.

- `fmin`/`fmax`는 사람 목소리~일반 악기 범위(C2~C7)로.
- 무성 구간은 NaN으로 들어오므로 유지한 채 반환. 후속 처리에서 필요 시 보간.
- 반올림한 반음(semitone) 궤적과 로그-스케일 유지 주파수를 함께 반환.

In [ ]:
def extract_pitch(y: np.ndarray, sr: int) -> Dict:
    f0, voiced_flag, voiced_prob = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
        sr=sr,
        frame_length=2048,
    )
    times = librosa.times_like(f0, sr=sr)

    # 반음 단위(MIDI) 변환 — 무성/결측은 NaN 유지
    with np.errstate(invalid="ignore", divide="ignore"):
        midi = librosa.hz_to_midi(f0)

    valid = ~np.isnan(f0)
    summary = {}
    if valid.any():
        summary = {
            "f0_median_hz": float(np.nanmedian(f0)),
            "midi_median": float(np.nanmedian(midi)),
            "midi_range": float(np.nanmax(midi) - np.nanmin(midi)),
            "voiced_ratio": float(valid.mean()),
        }

    return {
        "times": times,
        "f0_hz": f0,
        "midi": midi,
        "voiced_prob": voiced_prob,
        "summary": summary,
    }

## 3. 섹션 분할 — 두 종류 경계

서로 다른 질문엔 서로 다른 분할이 필요합니다.

**(a) 구조 경계** — "벌스/코러스가 어디서 반복되는가?"
McFee & Ellis(2014)의 **라플라시안 스펙트럴 분해**를 씁니다. 재발 행렬(recurrence) + 국소 유사도 그래프를 결합해 정규화 라플라시안을 만들고, 첫 몇 개의 고유벡터로 프레임을 클러스터링. Agglomerative보다 반복 구조(A-B-A-B-C-A…)를 훨씬 잘 잡습니다.

**(b) 분위기 경계** — "분위기가 언제 바뀌는가?"
valence/energy/tension을 프레임 단위로 계속 계산한 뒤, 슬라이딩 윈도우로 앞뒤 창의 평균 차이(유클리드 거리)를 **novelty curve**로 만들고 피크를 뽑습니다. 구조가 반복되는 동안 분위기만 서서히 바뀌는 구간(예: 빌드업)도 잡힙니다.

두 경계가 서로 다를 수 있는 게 포인트예요 — 같은 시간축에 겹쳐 보면 "구조는 그대로인데 분위기만 바뀌는 지점" 같은 게 드러납니다.

In [ ]:
from scipy.sparse.csgraph import laplacian as csgraph_laplacian
from sklearn.cluster import KMeans


def segment_song(y: np.ndarray, sr: int, min_segment_sec: float = 8.0) -> Dict:
    """라플라시안 스펙트럴 분해로 구조 경계를 뽑는다 (McFee & Ellis 2014).

    min_segment_sec: 이보다 짧은 구간은 인접 구간으로 흡수. 인트로처럼 악기가
    하나씩 들어올 때 라벨이 깜빡여 경계가 난립하는 걸 막는다.
    """
    duration = librosa.get_duration(y=y, sr=sr)

    tempo, beats = librosa.beat.beat_track(y=y, sr=sr, trim=False)
    tempo_val = float(np.atleast_1d(tempo)[0])

    # 비트 동기 특징 (chroma = 화성, mfcc = 음색)
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    if len(beats) < 8:
        # 비트가 너무 적으면 프레임 기반 폴백
        beats = np.arange(0, chroma.shape[1], 8)

    Csync = librosa.util.sync(chroma, beats, aggregate=np.median)
    Msync = librosa.util.sync(mfcc, beats, aggregate=np.mean)

    n = Csync.shape[1]
    if n < 4:
        # 너무 짧으면 그냥 통짜 섹션
        return {
            "tempo": tempo_val,
            "boundary_times": [0.0, float(duration)],
            "labels": [0],
        }

    # (1) 화성 재발 그래프 — 멀리 떨어진 비슷한 구간을 잇는다
    R = librosa.segment.recurrence_matrix(
        Csync, width=3, mode="affinity", sym=True,
    )
    # 대각선 스무딩 (재발 링크를 시간적으로 평활)
    df = librosa.segment.timelag_filter(median_filter)
    R = df(R, size=(1, 7))

    # (2) 음색 국소 유사도 — 인접 비트의 MFCC 거리
    path_distance = np.sum(np.diff(Msync, axis=1) ** 2, axis=0)
    sigma = np.median(path_distance) + 1e-9
    path_sim = np.exp(-path_distance / sigma)
    R_path = np.diag(path_sim, k=1) + np.diag(path_sim, k=-1)

    # (3) 두 그래프 결합 (McFee의 balancing)
    deg_path = np.sum(R_path, axis=1)
    deg_rec = np.sum(R, axis=1)
    mu = deg_path.dot(deg_path + deg_rec) / (np.sum((deg_path + deg_rec) ** 2) + 1e-9)
    A = mu * R + (1 - mu) * R_path

    # (4) 정규화 라플라시안 → 고유분해
    L, _ = csgraph_laplacian(A, normed=True, return_diag=True)
    try:
        evals, evecs = np.linalg.eigh(L)
    except np.linalg.LinAlgError:
        evals, evecs = np.linalg.eigh(L + 1e-6 * np.eye(L.shape[0]))

    # 곡 길이 기반 k
    k = int(np.clip(round(duration / 25.0), 3, 10))
    k = min(k, n - 1)

    # 첫 k개 고유벡터 → 정규화 → KMeans
    X = evecs[:, :k]
    X = librosa.util.normalize(X, norm=2, axis=1)
    km = KMeans(n_clusters=k, n_init=10, random_state=0)
    labels_beat = km.fit_predict(X)

    # 라벨 평활화 — 인트로 깜빡임을 클러스터 단계에서 흡수 (약 한 마디)
    labels_beat = median_filter(labels_beat, size=9, mode="nearest")

    beat_times = librosa.frames_to_time(beats, sr=sr)
    beat_times = np.concatenate([beat_times, [duration]])

    # 라벨이 바뀌는 위치가 경계
    change = np.flatnonzero(np.diff(labels_beat)) + 1
    boundary_beats = np.unique(np.concatenate([[0], change, [n]])).tolist()

    def _beat_time(bi):
        return float(beat_times[min(bi, len(beat_times) - 1)])

    def _seg_label(i):
        return int(labels_beat[min(boundary_beats[i], n - 1)])

    # 최소 길이 병합 — 임계 미만 구간은 인접 구간 중 더 가까운 쪽으로 흡수
    changed = True
    while changed and len(boundary_beats) > 2:
        changed = False
        for i in range(len(boundary_beats) - 1):
            start_t = _beat_time(boundary_beats[i])
            end_t = _beat_time(boundary_beats[i + 1])
            if end_t - start_t >= min_segment_sec:
                continue
            cur = _seg_label(i)
            prev_lbl = _seg_label(i - 1) if i > 0 else None
            next_lbl = _seg_label(i + 1) if i + 1 < len(boundary_beats) - 1 else None
            if prev_lbl is None and next_lbl is None:
                break
            # 같은 라벨이 있으면 그쪽, 없으면 더 긴 이웃 쪽으로 흡수
            if prev_lbl == cur:
                boundary_beats.pop(i)
            elif next_lbl == cur:
                boundary_beats.pop(i + 1)
            elif prev_lbl is None:
                boundary_beats.pop(i + 1)
            elif next_lbl is None:
                boundary_beats.pop(i)
            else:
                prev_len = start_t - _beat_time(boundary_beats[i - 1])
                next_len = _beat_time(boundary_beats[i + 2]) - end_t
                if prev_len >= next_len:
                    boundary_beats.pop(i)
                else:
                    boundary_beats.pop(i + 1)
            changed = True
            break

    boundary_times = [_beat_time(bi) for bi in boundary_beats]
    boundary_times = sorted(set(round(t, 3) for t in boundary_times))

    # 각 구간의 라벨 (첫 비트의 클러스터)
    seg_labels = []
    for i in range(len(boundary_beats) - 1):
        start_b = boundary_beats[i]
        seg_labels.append(int(labels_beat[min(start_b, n - 1)]))

    return {
        "tempo": tempo_val,
        "boundary_times": boundary_times,
        "labels": seg_labels,
    }

## 4. 분위기 분석 + 전환부 감지

곡 전체에 걸쳐 프레임 단위 저수준 특징을 뽑고, 섹션별로 집계해 해석 가능한 라벨을 붙입니다. 학습된 분류기가 없으므로 **규칙 기반 휴리스틱**으로 대체 — MER(music emotion recognition) 논문들에서 valence/arousal의 주요 상관자로 쓰는 특징들을 조합.

사용 특징:
- **RMS 에너지** — 라우드니스(arousal ↑)
- **Spectral centroid** — 밝기(valence/arousal ↑)
- **Spectral rolloff & flatness** — 텍스처 복잡도
- **Onset strength** — 리듬 밀도(arousal ↑)
- **Chroma 기반 major/minor 추정** — valence 극성
- **Tonnetz dissonance proxy** — tension

각 특징을 곡 내에서 z-score 정규화한 뒤, valence/energy/tension 세 축으로 집계합니다. 라벨은 축 조합으로 부여(예: 높은 energy + 높은 valence → 'energetic/bright'). 전환부는 섹션 경계에서 인접 섹션 간 이 축들의 유클리드 거리가 임계 이상인 지점을 리포트.

학습 기반 분류기가 아니므로 절대값이 아니라 **곡 내부 상대 비교**로 해석하세요.

In [ ]:
from scipy.signal import find_peaks


def _zscore(x: np.ndarray) -> np.ndarray:
    mu = np.nanmean(x)
    sd = np.nanstd(x) + 1e-9
    return (x - mu) / sd


_MAJOR_PROFILE = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
_MINOR_PROFILE = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])


def _estimate_mode(chroma: np.ndarray) -> float:
    """간이 major/minor 상관 — +1=major, -1=minor."""
    mean_chroma = chroma.mean(axis=1)
    if mean_chroma.sum() < 1e-6:
        return 0.0
    best_major = max(np.corrcoef(np.roll(mean_chroma, -i), _MAJOR_PROFILE)[0, 1] for i in range(12))
    best_minor = max(np.corrcoef(np.roll(mean_chroma, -i), _MINOR_PROFILE)[0, 1] for i in range(12))
    return float(best_major - best_minor)


def _label_mood(valence: float, energy: float, tension: float) -> str:
    v = "bright" if valence > 0.2 else ("dark" if valence < -0.2 else "neutral")
    e = "high-energy" if energy > 0.3 else ("calm" if energy < -0.3 else "moderate")
    parts = [e, v]
    if tension > 0.5:
        parts.append("tense")
    elif tension < -0.5:
        parts.append("relaxed")
    return " / ".join(parts)


def compute_mood_frames(y: np.ndarray, sr: int, hop: int = 512) -> Dict:
    """프레임 단위 valence/energy/tension 궤적."""
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
    flatness = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop)
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=hop)
    tonnetz = librosa.feature.tonnetz(chroma=chroma, sr=sr)

    N = len(rms)
    onset_env = onset_env[:N]
    chroma = chroma[:, :N]

    z_rms = _zscore(rms)
    z_centroid = _zscore(centroid[:N])
    z_onset = _zscore(onset_env)
    z_flatness = _zscore(flatness[:N])
    z_tonnetz = _zscore(tonnetz[:, :N].std(axis=0))

    # 프레임 단위 mode — 로컬 윈도우 chroma로 대략 추정
    win = max(1, int(sr / hop * 2))  # 약 2초 윈도우
    mode_trace = np.zeros(N)
    for i in range(N):
        lo = max(0, i - win)
        hi = min(N, i + win)
        mode_trace[i] = _estimate_mode(chroma[:, lo:hi])

    energy = 0.6 * z_rms + 0.4 * z_onset
    brightness = z_centroid
    valence = 0.7 * mode_trace + 0.3 * np.tanh(brightness)
    tension = 0.5 * z_tonnetz + 0.5 * z_flatness

    frame_times = librosa.frames_to_time(np.arange(N), sr=sr, hop_length=hop)
    return {
        "times": frame_times,
        "valence": valence,
        "energy": energy,
        "tension": tension,
        "brightness": brightness,
        "chroma": chroma,
        "hop": hop,
    }


def summarize_sections(mood_frames: Dict, boundary_times: List[float]) -> List[Dict]:
    """구조 경계 내부의 평균 분위기 → 섹션 요약."""
    times = mood_frames["times"]
    results = []
    for idx, (start, end) in enumerate(zip(boundary_times[:-1], boundary_times[1:])):
        mask = (times >= start) & (times < end)
        if not mask.any():
            continue
        v = float(mood_frames["valence"][mask].mean())
        e = float(mood_frames["energy"][mask].mean())
        t = float(mood_frames["tension"][mask].mean())
        results.append({
            "section_idx": idx,
            "start": float(start),
            "end": float(end),
            "duration": round(end - start, 2),
            "valence": round(v, 3),
            "energy": round(e, 3),
            "tension": round(t, 3),
            "mood_label": _label_mood(v, e, t),
        })
    return results


def mood_novelty_boundaries(mood_frames: Dict, window_sec: float = 4.0, min_gap_sec: float = 6.0) -> Dict:
    """분위기 novelty curve에서 피크를 뽑아 분위기 경계로 반환."""
    times = mood_frames["times"]
    if len(times) < 4:
        return {"times": [], "novelty": np.zeros_like(times), "frame_times": times}

    V = np.stack([mood_frames["valence"], mood_frames["energy"], mood_frames["tension"]], axis=0)
    dt = float(np.median(np.diff(times))) if len(times) > 1 else 0.05
    w = max(4, int(round(window_sec / dt)))
    min_distance = max(1, int(round(min_gap_sec / dt)))

    N = V.shape[1]
    novelty = np.zeros(N)
    for i in range(w, N - w):
        left = V[:, i - w : i].mean(axis=1)
        right = V[:, i : i + w].mean(axis=1)
        novelty[i] = np.linalg.norm(right - left)

    # 정규화 후 피크
    if novelty.max() > 0:
        novelty = novelty / novelty.max()
    peaks, _ = find_peaks(novelty, distance=min_distance, prominence=0.2)
    return {
        "times": [float(times[p]) for p in peaks],
        "novelty": novelty,
        "frame_times": times,
    }


def analyze_mood(y: np.ndarray, sr: int, boundary_times: List[float]) -> Dict:
    """프레임 단위 궤적 + 구조 경계 요약 + 분위기 경계를 한번에."""
    frames = compute_mood_frames(y, sr)
    sections = summarize_sections(frames, boundary_times)
    novelty = mood_novelty_boundaries(frames)
    return {
        "frames": frames,
        "sections": sections,
        "mood_boundaries": novelty,
    }

## 5. 파이프라인 래퍼

파일 경로 하나만 넣으면 세 분석을 모두 수행.

In [ ]:
def analyze_track(path: str) -> Dict:
    audio = load_audio(path)
    y_mono = audio["mono"]
    sr = audio["sr"]
    duration = librosa.get_duration(y=y_mono, sr=sr)

    pitch = extract_pitch(y_mono, sr)
    seg = segment_song(y_mono, sr)
    mood = analyze_mood(y_mono, sr, seg["boundary_times"])

    return {
        "path": path,
        "duration_sec": round(float(duration), 2),
        "sr": sr,
        "tempo_bpm": round(seg["tempo"], 2),
        "pitch": pitch,
        "segmentation": seg,
        "mood": mood,
        "_audio": {
            "left": audio["left"],
            "right": audio["right"],
            "mono": y_mono,
            "sr": sr,
        },
    }

## 6. 시각화 & 리포트

파형 + 섹션 경계 + 피치 궤적을 한 그림에 겹치고, 섹션별 분위기 표를 출력.

In [ ]:
def plot_report(result: Dict):
    y = result["_audio"]["mono"]
    sr = result["_audio"]["sr"]
    pitch = result["pitch"]
    struct_bounds = result["segmentation"]["boundary_times"]
    mood_bounds = result["mood"]["mood_boundaries"]["times"]
    novelty = result["mood"]["mood_boundaries"]["novelty"]
    nov_times = result["mood"]["mood_boundaries"]["frame_times"]

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

    librosa.display.waveshow(y, sr=sr, ax=axes[0], alpha=0.5)
    for b in struct_bounds:
        axes[0].axvline(b, color="crimson", lw=1.2, ls="--", label="_")
    for b in mood_bounds:
        axes[0].axvline(b, color="royalblue", lw=1.2, ls=":", label="_")
    axes[0].set_title("Waveform — red dashed: structural · blue dotted: mood")

    axes[1].plot(pitch["times"], pitch["f0_hz"], lw=1)
    axes[1].set_yscale("log")
    axes[1].set_ylabel("f0 (Hz)")
    axes[1].set_title("Pitch contour")
    for b in struct_bounds:
        axes[1].axvline(b, color="crimson", lw=1, ls="--", alpha=0.4)
    for b in mood_bounds:
        axes[1].axvline(b, color="royalblue", lw=1, ls=":", alpha=0.4)

    axes[2].plot(nov_times, novelty, color="royalblue", lw=1)
    axes[2].set_ylabel("mood novelty")
    axes[2].set_xlabel("time (s)")
    axes[2].set_title("Mood novelty curve (valence/energy/tension)")
    for b in mood_bounds:
        axes[2].axvline(b, color="royalblue", lw=1, ls=":", alpha=0.6)

    plt.tight_layout()
    plt.show()


def print_report(result: Dict):
    print(f"File      : {result['path']}")
    print(f"Duration  : {result['duration_sec']} s")
    print(f"Tempo     : {result['tempo_bpm']} BPM")
    ps = result["pitch"]["summary"]
    if ps:
        print(f"Pitch     : median {ps['f0_median_hz']:.1f} Hz, voiced {ps['voiced_ratio']*100:.0f}%, range {ps['midi_range']:.1f} semitones")
    print()
    print("Structural sections:")
    print(f"  {'#':>2}  {'start':>7}  {'end':>7}  {'val':>6}  {'eng':>6}  {'ten':>6}  mood")
    for s in result["mood"]["sections"]:
        print(f"  {s['section_idx']:>2}  {s['start']:>7.2f}  {s['end']:>7.2f}  "
              f"{s['valence']:>6.2f}  {s['energy']:>6.2f}  {s['tension']:>6.2f}  {s['mood_label']}")
    print()
    mb = result["mood"]["mood_boundaries"]["times"]
    if mb:
        print(f"Mood boundaries ({len(mb)}): " + ", ".join(f"{t:.1f}s" for t in mb))
    else:
        print("No mood boundaries detected.")

## 7. 실행

아래 경로를 분석하고 싶은 파일로 바꾸고 실행하세요.

In [ ]:
TRACK_PATH = "samples/your_audio.flac"  # .wav / .flac / .mp3 — samples/에 파일 넣고 경로 수정

result = analyze_track(TRACK_PATH)
print_report(result)
plot_report(result)

## 8. 재생하면서 보기

오디오 플레이어를 위에, 파형/피치 플롯을 아래에 띄웁니다. 플롯 x축(초) 눈금을 보면서 플레이어 위치를 맞춰 들으세요. 섹션 경계선 위에 라벨이 붙어있어 어느 구간에서 어떤 분위기가 나오는지 귀로 확인하기 좋습니다.

In [ ]:
from IPython.display import Audio, display


def listen_and_view(result: Dict):
    y = result["_audio"]["mono"]
    sr = result["_audio"]["sr"]
    pitch = result["pitch"]
    sections = result["mood"]["sections"]
    struct_bounds = result["segmentation"]["boundary_times"]
    mood_bounds = result["mood"]["mood_boundaries"]["times"]
    novelty = result["mood"]["mood_boundaries"]["novelty"]
    nov_times = result["mood"]["mood_boundaries"]["frame_times"]
    duration = result["duration_sec"]

    # 스테레오 재생
    y_stereo = np.stack([result["_audio"]["left"], result["_audio"]["right"]])
    display(Audio(data=y_stereo, rate=sr))

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

    librosa.display.waveshow(y, sr=sr, ax=axes[0], alpha=0.5)
    axes[0].set_title(f"{os.path.basename(result['path'])}  ·  {duration:.1f}s  ·  {result['tempo_bpm']} BPM   (red --: structural, blue ··: mood)")
    axes[0].set_ylabel("amp")

    # 구조 섹션 음영 + 라벨
    cmap = plt.get_cmap("tab20")
    for s in sections:
        color = cmap(s["section_idx"] % 20)
        axes[0].axvspan(s["start"], s["end"], color=color, alpha=0.12)
        mid = (s["start"] + s["end"]) / 2
        axes[0].text(
            mid, axes[0].get_ylim()[1] * 0.85,
            f"#{s['section_idx']}\n{s['mood_label']}",
            ha="center", va="top", fontsize=8,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.7),
        )
    for b in struct_bounds:
        axes[0].axvline(b, color="crimson", lw=1.2, ls="--", alpha=0.8)
    for b in mood_bounds:
        axes[0].axvline(b, color="royalblue", lw=1.2, ls=":", alpha=0.9)

    axes[1].plot(pitch["times"], pitch["f0_hz"], lw=1)
    axes[1].set_yscale("log")
    axes[1].set_ylabel("f0 (Hz)")
    axes[1].set_title("Pitch contour (pYIN)")
    for b in struct_bounds:
        axes[1].axvline(b, color="crimson", lw=1, ls="--", alpha=0.4)
    for b in mood_bounds:
        axes[1].axvline(b, color="royalblue", lw=1, ls=":", alpha=0.5)

    axes[2].plot(nov_times, novelty, color="royalblue", lw=1)
    axes[2].set_ylabel("mood novelty")
    axes[2].set_xlabel("time (s)")
    axes[2].set_title("Mood novelty curve")
    for b in mood_bounds:
        axes[2].axvline(b, color="royalblue", lw=1, ls=":", alpha=0.6)

    tick_step = 5 if duration < 120 else 10
    axes[2].set_xticks(np.arange(0, duration + 1, tick_step))
    axes[2].set_xlim(0, duration)
    axes[2].grid(axis="x", alpha=0.3)

    plt.tight_layout()
    plt.show()


listen_and_view(result)

## 9. 결과 저장

L/R 채널을 개별 wav로 저장하고, 분석 데이터(섹션·피치·분위기)를 CSV로 내보냅니다.

출력 디렉토리는 원본 파일과 같은 위치에 `{파일명}_analysis/` 로 생성됩니다.

In [ ]:
import csv
import soundfile as sf


def export_result(result: Dict, out_dir: str = None):
    """L/R wav + 분석 CSV를 저장한다."""
    path = result["path"]
    stem = os.path.splitext(os.path.basename(path))[0]
    if out_dir is None:
        out_dir = os.path.join(os.path.dirname(path), f"{stem}_analysis")
    os.makedirs(out_dir, exist_ok=True)

    sr = result["_audio"]["sr"]

    # ── L/R 채널 wav ──
    sf.write(os.path.join(out_dir, f"{stem}_L.wav"), result["_audio"]["left"], sr)
    sf.write(os.path.join(out_dir, f"{stem}_R.wav"), result["_audio"]["right"], sr)

    # ── 섹션 요약 CSV ──
    sections = result["mood"]["sections"]
    sec_path = os.path.join(out_dir, f"{stem}_sections.csv")
    with open(sec_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=[
            "section_idx", "start", "end", "duration",
            "valence", "energy", "tension", "mood_label",
        ])
        w.writeheader()
        w.writerows(sections)

    # ── 피치 궤적 CSV ──
    pitch = result["pitch"]
    pitch_path = os.path.join(out_dir, f"{stem}_pitch.csv")
    with open(pitch_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["time", "f0_hz", "midi", "voiced_prob"])
        for t, hz, m, vp in zip(
            pitch["times"], pitch["f0_hz"], pitch["midi"], pitch["voiced_prob"]
        ):
            w.writerow([round(t, 4), round(hz, 3) if not np.isnan(hz) else "",
                        round(m, 3) if not np.isnan(m) else "",
                        round(vp, 4)])

    # ── 프레임 단위 분위기 궤적 CSV ──
    mf = result["mood"]["frames"]
    mood_path = os.path.join(out_dir, f"{stem}_mood_frames.csv")
    with open(mood_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["time", "valence", "energy", "tension", "brightness"])
        for i in range(len(mf["times"])):
            w.writerow([
                round(mf["times"][i], 4),
                round(float(mf["valence"][i]), 4),
                round(float(mf["energy"][i]), 4),
                round(float(mf["tension"][i]), 4),
                round(float(mf["brightness"][i]), 4),
            ])

    # ── 경계 CSV (구조 + 분위기) ──
    bounds_path = os.path.join(out_dir, f"{stem}_boundaries.csv")
    struct_set = set(result["segmentation"]["boundary_times"])
    mood_set = set(result["mood"]["mood_boundaries"]["times"])
    all_bounds = sorted(struct_set | mood_set)
    with open(bounds_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["time", "type"])
        for t in all_bounds:
            types = []
            if t in struct_set:
                types.append("structural")
            if t in mood_set:
                types.append("mood")
            w.writerow([round(t, 3), "+".join(types)])

    # ── 메타 CSV ──
    meta_path = os.path.join(out_dir, f"{stem}_meta.csv")
    with open(meta_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["key", "value"])
        w.writerow(["file", path])
        w.writerow(["duration_sec", result["duration_sec"]])
        w.writerow(["sr", sr])
        w.writerow(["tempo_bpm", result["tempo_bpm"]])
        ps = result["pitch"]["summary"]
        for k, v in ps.items():
            w.writerow([k, round(v, 4)])

    print(f"Exported to {out_dir}/")
    print(f"  {stem}_L.wav / {stem}_R.wav")
    print(f"  {stem}_sections.csv      ({len(sections)} sections)")
    print(f"  {stem}_pitch.csv         ({len(pitch['times'])} frames)")
    print(f"  {stem}_mood_frames.csv   ({len(mf['times'])} frames)")
    print(f"  {stem}_boundaries.csv    ({len(all_bounds)} boundaries)")
    print(f"  {stem}_meta.csv")
    return out_dir


export_result(result)